# Test Henry Scenario Dataset

This notebook validates the custom dataset implementation in `src/data/henry_scenario_dataset.py`.

It checks:
- deterministic 80/20 run split
- no run overlap between train and validation
- per-sample tensor shapes `(C_in, H, W)` and `(C_out, H, W)`
- dataloader batch shapes and shuffle behavior

In [34]:
from pathlib import Path
import sys

import torch

# Ensure workspace root is importable in notebook context.
cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "src").exists() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.henry_scenario_dataset import HenryScenarioDataset, create_henry_dataloaders

scenario_dir = Path(
    "/Users/akap5486/Projects/groundwater/data/henry_data/all_scenarios/coupling_scenarios/scenario_01"
)

print("cwd:", cwd)
print("project_root:", project_root)
print("scenario_dir exists:", scenario_dir.exists())
print("torch:", torch.__version__)

cwd: /Users/akap5486/Projects/groundwater/synthetic_problems/gw_henry_fno/notebooks
project_root: /Users/akap5486/Projects/groundwater/synthetic_problems/gw_henry_fno
scenario_dir exists: True
torch: 2.11.0


In [35]:
seed = 42
train_ratio = 0.8

train_ds = HenryScenarioDataset(
    scenario_dir=scenario_dir,
    split="train",
    train_ratio=train_ratio,
    seed=seed,
)
val_ds = HenryScenarioDataset(
    scenario_dir=scenario_dir,
    split="val",
    train_ratio=train_ratio,
    seed=seed,
)

train_runs = set(train_ds.run_names)
val_runs = set(val_ds.run_names)

print("train runs:", len(train_runs))
print("val runs:", len(val_runs))
print("run overlap:", len(train_runs.intersection(val_runs)))

assert len(train_runs) > 0 and len(val_runs) > 0, "Empty split detected"
assert len(train_runs.intersection(val_runs)) == 0, "Train/val run overlap found"

train runs: 64
val runs: 17
run overlap: 0


In [36]:
x, y = train_ds[0]
print("single sample shapes:", tuple(x.shape), tuple(y.shape))

assert x.ndim == 3, f"Expected input ndim=3, got {x.ndim}"
assert y.ndim == 3, f"Expected output ndim=3, got {y.ndim}"
assert x.shape[1:] == y.shape[1:], "Input/output spatial shapes do not match"

# Expected for current scenario_01 data
assert tuple(x.shape) == (8, 40, 80), f"Unexpected input shape: {tuple(x.shape)}"
assert tuple(y.shape) == (2, 40, 80), f"Unexpected output shape: {tuple(y.shape)}"

single sample shapes: (8, 40, 80) (2, 40, 80)


In [37]:
len(train_ds), len(val_ds)

(6336, 1683)

In [38]:
batch_size = 256
train_loader, val_loader = create_henry_dataloaders(
    scenario_dir=scenario_dir,
    batch_size=batch_size,
    train_ratio=train_ratio,
    seed=seed,
    num_workers=0,
    pin_memory=False,
)

xb, yb = next(iter(train_loader))
print("train batch shapes:", tuple(xb.shape), tuple(yb.shape))
print("train sampler:", type(train_loader.sampler).__name__)
print("val sampler:", type(val_loader.sampler).__name__)

assert xb.ndim == 4 and yb.ndim == 4, "Expected batched tensors with ndim=4"
assert xb.shape[0] == batch_size and yb.shape[0] == batch_size, "Unexpected batch size"
assert tuple(xb.shape[1:]) == (8, 40, 80), f"Unexpected train input batch shape: {tuple(xb.shape)}"
assert tuple(yb.shape[1:]) == (2, 40, 80), f"Unexpected train output batch shape: {tuple(yb.shape)}"
assert type(train_loader.sampler).__name__ == "RandomSampler", "Train loader must be shuffled"
assert type(val_loader.sampler).__name__ == "SequentialSampler", "Val loader must be unshuffled"

print("All dataset tests passed.")

train batch shapes: (256, 8, 40, 80) (256, 2, 40, 80)
train sampler: RandomSampler
val sampler: SequentialSampler
All dataset tests passed.


In [39]:
import torch
import torch.nn.functional as F

from src.neuralop import FNO

def get_best_device() -> torch.device:
    # Prefer CUDA when available, then MPS, and finally CPU.
    if torch.cuda.is_available():
        return torch.device("cuda")
        if torch.backends.mps.is_available():
            return torch.device("mps")
    return torch.device("cpu")


device = get_best_device()

# Build a small 2D FNO matching the dataset channel dimensions.

for hidden_channels in [8, 16, 32, 64, 128]:
    model = FNO(
        n_modes=(8, 16),
        hidden_channels=hidden_channels,
        in_channels=xb.shape[1],
        out_channels=yb.shape[1],
        n_layers=4,
    ).to(device)

    # number of parameters in FNO
    num_params = sum(p.numel() for p in model.parameters())
    print(f"FNO parameter count: {num_params}")

FNO parameter count: 44330
FNO parameter count: 159826
FNO parameter count: 613538
FNO parameter count: 2411842
FNO parameter count: 9571970


In [40]:

%%time
xb_device = xb.to(device)
yb_device = yb.to(device)

model.eval()
with torch.no_grad():
    pred = model(xb_device)

print("device:", device)
print("pred shape:", tuple(pred.shape))
print("target shape:", tuple(yb_device.shape))

assert pred.shape == yb_device.shape, f"Prediction/target shape mismatch: {tuple(pred.shape)} vs {tuple(yb_device.shape)}"

mse = F.mse_loss(pred, yb_device)
print("mse:", float(mse))

/Users/akap5486/Projects/groundwater/synthetic_problems/gw_henry_fno/.venv/lib/python3.14/site-packages/tltorch/factorized_tensors/factorized_tensors.py:66: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/python_variable_indexing.cpp:353.)
  return self.__class__(self.tensor[indices])
/Users/akap5486/Projects/groundwater/synthetic_problems/gw_henry_fno/src/neuralop/conv.py:201: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a differ

device: cpu
pred shape: (256, 2, 40, 80)
target shape: (256, 2, 40, 80)
mse: 91.541015625
CPU times: user 15 s, sys: 1.01 s, total: 16 s
Wall time: 2.95 s
